# 09 — Final held-out evaluation

**Objective.** Measure the selected system once, on documents never used in development. B1 is ColQwen2 top 4 → `gpt-6-luna` with contract b1. The evaluation:
- estimates B1's accuracy (**C1**);
- re-tests the project's central result, visual retrieval against OCR + BM25 retrieval with the same reader (**C2**, confirmatory);
- reports E9's adaptive policy A1 as a secondary contrast (**C3**).

The protocol is frozen in `experiments/specs/final_evaluation_v1.json` (commit `1c54a3e`). The owner approved it on 2026-11-25, before any test-pool document was downloaded.

**Stages and modes** (`VISUALDOC_FINAL_MODE`):

| Mode | What it does | Reads questions? | Paid? |
|---|---|---|---|
| `offline` (default) | reports the state of every stage; evaluates a completed run | only once the index is frozen | no |
| `index` | downloads the 111 pool PDFs, extracts text (OCR below 30 words), renders 768-px pages, runs the E5 parity gate, then encodes every page with ColQwen2 (resumable) | **no** | no |
| `queries` | after the index manifest is committed: loads the pool questions and encodes them | yes | no |
| `smoke` / `full` | the reader runs (smoke on development questions only) | yes | yes |

**Inputs.**
- The frozen spec.
- `resources/mmlongbench_doc_documents.json` (document names).
- The MMLongBench-Doc PDFs at revision `2ff6aa92…`, downloaded here.
- The ColQwen2 snapshot `ddc07d23…`, in the Hugging Face cache outside the repository.
- The E5 encoder configuration and cache, used for the parity gate.

**Outputs.**
- `data/processed/mmlongbench_doc/final/` (document and page tables, the encoding log, the index manifest).
- `data/cache/final/` (page embeddings).
- Later, the run and its evaluation.

**Next step.** Commit the index manifest; then the `queries` mode.

## Method (indexing stage)

**Same pipeline as E5, unchanged.** Every page is represented in two ways:
- **Text:** the native text layer, or full-page Tesseract OCR at 150 dpi when it has fewer than 30 words. This text feeds BM25 for T1.
- **Image:** the whole page fitted in a 768 × 1536 box as an RGB PNG. This is the image ColQwen2 indexes and the reader sees.

ColQwen2 maps an image to a matrix of 128-dimensional vectors, one per visual token, and a question to one vector per query token. Retrieval scores a page by late interaction (MaxSim): $s(q,d)=\sum_{i}\max_j \langle q_i, d_j\rangle$.

**Two passes, now in chunks.** On an 8 GB machine, E5 ran the vision tower over every page first (pass 1), then the language model with the vision tower replaced by the saved features (pass 2). E5 proved this bitwise equal to the single pass.
- Pass-1 features take about 1.8 MB per page.
- For about 5,300 pages that would mean about 9.5 GB of transient disk.
- So the passes alternate over chunks of 200 pages. Each embedding is computed by exactly the same operations, so this changes only when the model is reloaded, not what is computed.

**Parity gate (frozen spec).** Before any pool page is encoded, two development pages and one development question are re-encoded from scratch. All three must be **bitwise equal** to the frozen E5 cache. This proves that the weights, processor, library versions and the chunked two-pass procedure reproduce E5 exactly. Otherwise the notebook stops.

**No question is read in this stage.** Indexing depends only on the documents. The pool questions are loaded only after the index manifest (every page's image hash and embedding hash) is committed, so retrieval cannot be adapted to the questions.

In [1]:
# Setup: paths, the frozen spec (must equal its freeze commit), mode, and small file utilities.
from pathlib import Path
from types import SimpleNamespace
import gc
import hashlib
import io
import json
import os
import subprocess
import time

import numpy as np
import pandas as pd
import pymupdf
from IPython.display import display

pd.set_option("display.width", 170)


def find_project_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise FileNotFoundError("Could not find the repository root")


PROJECT_ROOT = find_project_root()
SPEC_PATH = PROJECT_ROOT / "experiments" / "specs" / "final_evaluation_v1.json"
SPEC_FREEZE_COMMIT = "1c54a3e"
DOC_LIST_PATH = PROJECT_ROOT / "resources" / "mmlongbench_doc_documents.json"
MMLONG_DIR = PROJECT_ROOT / "data" / "external" / "mmlongbench_doc"
PDF_DIR = MMLONG_DIR / "documents"
TESSDATA_DIR = PROJECT_ROOT / "resources" / "tessdata"
IMAGE_DIR = PROJECT_ROOT / "data" / "cache" / "mmlongbench_doc" / "page_images" / "short768-long1536"
FINAL_DIR = PROJECT_ROOT / "data" / "processed" / "mmlongbench_doc" / "final"
E5_CACHE_DIR = PROJECT_ROOT / "data" / "cache" / "e5" / "colqwen2-v1.0-hf@ddc07d2317c8" / "af02fac2cb9b"
E5_FROZEN = PROJECT_ROOT / "experiments" / "frozen" / "e5"

REQUIRED = {SPEC_PATH: "restore the frozen spec from git", DOC_LIST_PATH: "restore resources/ from git",
            TESSDATA_DIR / "eng.traineddata": "the OCR model is tracked in git: restore resources/tessdata",
            E5_FROZEN / "SHA256SUMS": "restore experiments/frozen/e5 from git"}
missing = [f"{p.relative_to(PROJECT_ROOT)} ({fix})" for p, fix in REQUIRED.items() if not p.is_file()]
assert not missing, "Notebook 09 inputs missing:\n  " + "\n  ".join(missing)

git = lambda *args: subprocess.run(["git", *args], cwd=PROJECT_ROOT, capture_output=True, check=True).stdout
assert git("show", f"{SPEC_FREEZE_COMMIT}:experiments/specs/final_evaluation_v1.json") == SPEC_PATH.read_bytes(), \
    "the final-evaluation spec changed after its freeze commit: that is a new condition"
SPEC = json.loads(SPEC_PATH.read_text())
REVISION = SPEC["population"]["revision"]
MODE = os.environ.get("VISUALDOC_FINAL_MODE", "offline")
assert MODE in {"offline", "index", "queries", "smoke", "full"}, f"unknown VISUALDOC_FINAL_MODE {MODE!r}"
SETTINGS = {"ocr_min_native_words": 30, "ocr_dpi": 150, "image_short_px": 768, "image_long_px": 1536,
            "bm25_k1": 1.5, "bm25_b": 0.75, "encode_chunk_pages": 200}
FINAL_DIR.mkdir(parents=True, exist_ok=True)


def sha256_bytes(data):
    return hashlib.sha256(data).hexdigest()


def sha256_file(path, chunk=1 << 24):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(chunk), b""):
            digest.update(block)
    return digest.hexdigest()


def atomic_write_bytes(path, data):
    """Write to a temporary sibling, fsync, then rename: an interruption never leaves a half-written file."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(f".{path.name}.tmp-{os.getpid()}")
    with open(tmp, "wb") as handle:
        handle.write(data)
        handle.flush()
        os.fsync(handle.fileno())
    os.replace(tmp, path)


def remove_stale_temporaries(folder):
    stale = list(Path(folder).glob(".*.tmp-*")) if Path(folder).is_dir() else []
    for path in stale:
        path.unlink()
    return len(stale)


print(f"Spec equals commit {SPEC_FREEZE_COMMIT}. Mode: {MODE}. Outputs: {FINAL_DIR.relative_to(PROJECT_ROOT)}")

Spec equals commit 1c54a3e. Mode: offline. Outputs: data/processed/mmlongbench_doc/final


In [2]:
# The population, from document names only (frame metadata): 135 - 20 development - 4 touched = 111. No question is read.
doc_list = json.loads(DOC_LIST_PATH.read_text())
assert doc_list["revision"] == REVISION
ALL_DOCS = sorted(doc_list["documents"])
assert len(ALL_DOCS) == 135

e5_spec = json.loads((PROJECT_ROOT / "experiments" / "specs" / "e5_visual_retrieval_v1.json").read_text())
HELDOUT14 = sorted(e5_spec["populations"]["held_out"]["selected_documents"])
DEV6 = sorted(pd.read_csv(E5_FROZEN / "tables" / "per_question.csv").query("population == 'development'")["doc_id"].unique())
TOUCHED = sorted(SPEC["population"]["excluded_touched_documents"])
assert len(HELDOUT14) == 14 and len(DEV6) == 6 and len(TOUCHED) == 4
assert not (set(HELDOUT14) | set(DEV6)) & set(TOUCHED) and set(HELDOUT14) | set(DEV6) | set(TOUCHED) <= set(ALL_DOCS)
POOL_DOCS = sorted(set(ALL_DOCS) - set(HELDOUT14) - set(DEV6) - set(TOUCHED))
assert len(POOL_DOCS) == SPEC["population"]["documents"] == 111
print(f"Population: {len(POOL_DOCS)} documents (135 - 20 development - 4 touched). No question has been read.")

Population: 111 documents (135 - 20 development - 4 touched). No question has been read.


In [3]:
# Download the 111 pool PDFs at the pinned revision (index mode), and record size, SHA-256 and page count.
DOCS_TABLE = FINAL_DIR / "pool_documents.csv"


def local_pdf(doc_id):
    return PDF_DIR / doc_id


def materialised(path):
    """A real local file (not an iCloud placeholder that would read as empty)."""
    return path.is_file() and path.stat().st_size > 0 and not (os.stat(path).st_flags & 0x40000000)


if MODE == "index":
    from huggingface_hub import hf_hub_download
    rows, started = [], time.perf_counter()
    for i, doc_id in enumerate(POOL_DOCS, 1):
        if not materialised(local_pdf(doc_id)):
            # A file left by an earlier download may be an evicted iCloud placeholder (it reads as empty): force a fresh copy.
            hf_hub_download("yubo2333/MMLongBench-Doc", f"documents/{doc_id}", repo_type="dataset", revision=REVISION, local_dir=MMLONG_DIR,
                            force_download=local_pdf(doc_id).exists())
        path = local_pdf(doc_id)
        assert materialised(path), f"{doc_id}: download did not produce a local file"
        try:
            with pymupdf.open(path) as pdf:
                pages, encrypted = pdf.page_count, pdf.needs_pass
        except Exception as error:  # recorded, never silently dropped
            pages, encrypted = 0, None
        rows.append({"doc_id": doc_id, "bytes": path.stat().st_size, "sha256": sha256_file(path), "pages": pages, "encrypted": encrypted})
        if i % 20 == 0 or i == len(POOL_DOCS):
            print(f"  {i}/{len(POOL_DOCS)} documents, {time.perf_counter() - started:.0f} s", flush=True)
    atomic_write_bytes(DOCS_TABLE, pd.DataFrame(rows).to_csv(index=False).encode())
if DOCS_TABLE.is_file():
    pool_documents = pd.read_csv(DOCS_TABLE)
    assert sorted(pool_documents["doc_id"]) == POOL_DOCS
    unreadable = pool_documents[(pool_documents["pages"] < 1) | (pool_documents["encrypted"] == True)]
    print(f"{len(pool_documents)} pool PDFs: {pool_documents['bytes'].sum() / 1e9:.2f} GB, {int(pool_documents['pages'].sum())} pages "
          f"(per document: median {pool_documents['pages'].median():.0f}, max {pool_documents['pages'].max()}); unreadable: {len(unreadable)}.")
    display(pd.cut(pool_documents["pages"], [0, 50, 200, 10_000], labels=["<= 50", "51-200", "> 200"]).value_counts().sort_index().to_frame("documents"))
else:
    pool_documents = None
    print("Pool PDFs not downloaded yet (run with VISUALDOC_FINAL_MODE=index).")

Pool PDFs not downloaded yet (run with VISUALDOC_FINAL_MODE=index).


In [4]:
# Text (notebook 00's rule) and the 768-px render (E1 profile) of every pool page; one parquet per document, resumable.
PAGES_DIR = FINAL_DIR / "pages"


def page_text_record(page):
    native_text = page.get_text("text", sort=True)
    native_words = len(native_text.split())
    ocr_used = native_words < SETTINGS["ocr_min_native_words"]
    if ocr_used:
        textpage = page.get_textpage_ocr(language="eng", dpi=SETTINGS["ocr_dpi"], full=True, tessdata=str(TESSDATA_DIR))
        text = page.get_text("text", textpage=textpage, sort=True)
    else:
        text = native_text
    return {"native_words": native_words, "ocr_used": ocr_used, "text": text}


def render_answering_png(page):
    zoom = min(SETTINGS["image_short_px"] / min(page.rect.width, page.rect.height),
               SETTINGS["image_long_px"] / max(page.rect.width, page.rect.height))
    return page.get_pixmap(matrix=pymupdf.Matrix(zoom, zoom), alpha=False).tobytes("png")


def image_path(doc_id, page_number):
    return IMAGE_DIR / Path(doc_id).stem / f"page_{page_number:03d}.png"


# Determinism check on development pages first: a fresh render must equal E5's frozen bytes.
e5_manifest = pd.read_csv(E5_FROZEN / "tables" / "render_manifest_heldout.csv")
render_checked = []
for doc_id, group in e5_manifest.groupby("doc_id"):
    if materialised(local_pdf(doc_id)) and len(render_checked) < 2:  # iCloud placeholders cannot be read offline
        with pymupdf.open(local_pdf(doc_id)) as pdf:
            for row in group.head(3).itertuples():
                assert sha256_bytes(render_answering_png(pdf[row.page - 1])) == row.sha256, f"render differs from E5 for {doc_id} p.{row.page}"
        render_checked.append(doc_id)
assert render_checked, "no development PDF is available locally to check rendering determinism"
print(f"Rendering reproduces E5's frozen page bytes on 3 pages of each of {len(render_checked)} development documents.")

if MODE == "index":
    remove_stale_temporaries(PAGES_DIR)
    started, done = time.perf_counter(), 0
    for doc_id in POOL_DOCS:
        target = PAGES_DIR / f"{Path(doc_id).stem}.parquet"
        if target.is_file():
            continue
        rows = []
        with pymupdf.open(local_pdf(doc_id)) as pdf:
            for index, page in enumerate(pdf):
                png, out = render_answering_png(page), image_path(doc_id, index + 1)
                if not out.is_file() or out.read_bytes() != png:
                    atomic_write_bytes(out, png)
                width, height = int.from_bytes(png[16:20], "big"), int.from_bytes(png[20:24], "big")
                rows.append({"doc_id": doc_id, "page": index + 1, **page_text_record(page), "width_px": width, "height_px": height,
                             "sha256": sha256_bytes(png), "image_path": str(out.relative_to(PROJECT_ROOT))})
        buffer = io.BytesIO()
        pd.DataFrame(rows).to_parquet(buffer, index=False)
        atomic_write_bytes(target, buffer.getvalue())
        done += 1
        print(f"  pages: {doc_id[:50]} ({len(rows)} pages), {done} new documents, {time.perf_counter() - started:.0f} s", flush=True)

page_files = sorted(PAGES_DIR.glob("*.parquet")) if PAGES_DIR.is_dir() else []
pool_pages = pd.concat([pd.read_parquet(p) for p in page_files], ignore_index=True) if page_files else None
if pool_pages is not None:
    print(f"Page records for {pool_pages['doc_id'].nunique()}/{len(POOL_DOCS)} documents: {len(pool_pages)} pages, "
          f"OCR on {int(pool_pages['ocr_used'].sum())}, {pool_pages['sha256'].nunique()} distinct images.")

Rendering reproduces E5's frozen page bytes on 3 pages of each of 2 development documents.


In [5]:
# The ColQwen2 encoder: configuration identical to E5's (checked field by field), weights verified by SHA-256.
ENCODE_MODES = {"index", "queries"}
E5_CONFIG = json.loads((E5_CACHE_DIR / "config.json").read_text()) if (E5_CACHE_DIR / "config.json").is_file() else None
FINAL_CACHE = PROJECT_ROOT / "data" / "cache" / "final" / "colqwen2-v1.0-hf@ddc07d2317c8" / "af02fac2cb9b"
ENCODING_LOG = FINAL_DIR / "encoding_log.csv"

if MODE in ENCODE_MODES:
    import torch
    import transformers
    from huggingface_hub import snapshot_download
    from PIL import Image
    from safetensors.torch import load_file, save_file
    from transformers import ColQwen2ForRetrieval, ColQwen2Processor
    import psutil

    assert E5_CONFIG is not None, "the E5 encoder cache (data/cache/e5/...) is needed for the identity and parity checks"
    MODEL_DIR = Path(snapshot_download(E5_CONFIG["repo"], revision=E5_CONFIG["revision"], local_files_only=True))
    DEVICE, DTYPE, ATTENTION = "mps", torch.bfloat16, "sdpa"
    processor = ColQwen2Processor.from_pretrained(MODEL_DIR)
    ENCODER_CONFIG = {
        "repo": E5_CONFIG["repo"], "revision": E5_CONFIG["revision"], "weights_sha256": sha256_file(MODEL_DIR / "model.safetensors"),
        "transformers": transformers.__version__, "torch": torch.__version__, "device": DEVICE, "dtype": "bfloat16",
        "attn_implementation": ATTENTION, "batch_size": 1, "page_image": "E1 profile: 768 x 1536 box, RGB PNG",
        "preprocessor_config_sha256": sha256_file(MODEL_DIR / "preprocessor_config.json"),
        "tokenizer_sha256": sha256_file(MODEL_DIR / "tokenizer.json"),
        "visual_prompt": processor.visual_prompt_prefix, "query_prefix": processor.query_prefix,
        "query_augmentation": f"10 x {processor.query_augmentation_token}",
        "embedding_rule": "model(**batch).embeddings; rows with attention_mask 0 removed; stored bf16",
    }
    differences = {k: (E5_CONFIG.get(k), ENCODER_CONFIG.get(k)) for k in set(E5_CONFIG) | set(ENCODER_CONFIG) if E5_CONFIG.get(k) != ENCODER_CONFIG.get(k)}
    assert not differences, f"the encoder is not E5's: {differences}"
    assert hashlib.sha256(json.dumps(ENCODER_CONFIG, sort_keys=True).encode()).hexdigest()[:12] == FINAL_CACHE.name
    for sub in ("pages", "queries", "vision_features", "parity"):
        (FINAL_CACHE / sub).mkdir(parents=True, exist_ok=True)
        remove_stale_temporaries(FINAL_CACHE / sub)
    atomic_write_bytes(FINAL_CACHE / "config.json", json.dumps(ENCODER_CONFIG, indent=1, sort_keys=True).encode())
    print(f"Encoder configuration equals E5's in all {len(ENCODER_CONFIG)} fields (weights {ENCODER_CONFIG['weights_sha256'][:12]}…). "
          f"Cache: {FINAL_CACHE.relative_to(PROJECT_ROOT)}")
else:
    print("Encoder not loaded (offline mode).")

Encoder not loaded (offline mode).


In [6]:
# The two passes of notebook 04, unchanged in substance, run over a list of page jobs (or questions) into a cache folder.
if MODE in ENCODE_MODES:
    process = psutil.Process()

    def atomic_save_tensors(tensors, path):
        tmp = path.with_name(f".{path.name}.tmp-{os.getpid()}")
        save_file(tensors, str(tmp))
        os.replace(tmp, path)

    def load_page_image(relative_path):
        with Image.open(PROJECT_ROOT / relative_path) as image:
            return image.convert("RGB")

    def unpadded_pixel_values(batch):
        # The library's own unpadding (ColQwen2ForRetrieval.forward): keep each image's h * w patch rows.
        grid = batch["image_grid_thw"]
        offsets = grid[:, 1] * grid[:, 2]
        mask = torch.arange(batch["pixel_values"].shape[1]).unsqueeze(0) < offsets.unsqueeze(1)
        return batch["pixel_values"][mask]

    class CachedVisionFeatures(torch.nn.Module):
        """Stands in for model.vlm.visual: returns the pass-1 output of the page being encoded."""
        def __init__(self):
            super().__init__()
            self.features = None

        def forward(self, pixel_values, grid_thw=None, return_dict=True, **kwargs):
            expected = int((grid_thw.prod(-1) // 4).sum())
            assert self.features is not None and self.features.shape[0] == expected, "cached features do not match this page"
            return SimpleNamespace(pooler_output=self.features)

    def encode_with(model, batch):
        embeddings = model(**batch).embeddings
        torch.mps.synchronize()
        keep = batch["attention_mask"][0].bool()
        return embeddings[0][keep].cpu().contiguous()

    def log_rows(rows):
        existing = ENCODING_LOG.read_bytes() if ENCODING_LOG.is_file() else b""
        atomic_write_bytes(ENCODING_LOG, existing + pd.DataFrame(rows).to_csv(index=False, header=not existing).encode())

    def pass1(jobs, cache):
        """Vision tower only, for pages without an embedding or saved features."""
        todo = [j for j in jobs if not (cache / "pages" / f"{j['sha256']}.safetensors").is_file()
                and not (cache / "vision_features" / f"{j['sha256']}.safetensors").is_file()]
        if not todo:
            return 0
        model = ColQwen2ForRetrieval.from_pretrained(MODEL_DIR, dtype=DTYPE, attn_implementation=ATTENTION).eval()
        visual = model.vlm.visual
        model.vlm.visual = None
        del model
        gc.collect()
        visual.to(DEVICE)
        rows = []
        with torch.inference_mode():
            for job in todo:
                t0 = time.perf_counter()
                batch = processor(images=[load_page_image(job["image_path"])], return_tensors="pt")
                features = visual(unpadded_pixel_values(batch).to(DEVICE), grid_thw=batch["image_grid_thw"].to(DEVICE), return_dict=True).pooler_output
                torch.mps.synchronize()
                atomic_save_tensors({"features": features.cpu().contiguous()}, cache / "vision_features" / f"{job['sha256']}.safetensors")
                rows.append({"pass": "vision", "key": job["sha256"], "doc_id": job["doc_id"], "page": job["page"],
                             "seconds": round(time.perf_counter() - t0, 3), "tokens": int(features.shape[0]), "rss_gb": round(process.memory_info().rss / 1e9, 2)})
        del visual
        gc.collect()
        torch.mps.empty_cache()
        log_rows(rows)
        return len(todo)

    def pass2(jobs, questions, cache):
        """The library's forward with the vision tower replaced by the pass-1 features; then any questions."""
        todo = [j for j in jobs if not (cache / "pages" / f"{j['sha256']}.safetensors").is_file()]
        qtodo = [q for q in questions if not (cache / "queries" / f"{sha256_bytes(q.encode())}.safetensors").is_file()]
        if not todo and not qtodo:
            return 0, 0
        model = ColQwen2ForRetrieval.from_pretrained(MODEL_DIR, dtype=DTYPE, attn_implementation=ATTENTION).eval()
        stub = CachedVisionFeatures()
        model.vlm.visual = stub
        gc.collect()
        model.to(DEVICE)
        rows = []
        with torch.inference_mode():
            for job in todo:
                t0 = time.perf_counter()
                batch = processor(images=[load_page_image(job["image_path"])], return_tensors="pt").to(DEVICE)
                feature_file = cache / "vision_features" / f"{job['sha256']}.safetensors"
                stub.features = load_file(str(feature_file))["features"].to(DEVICE)
                embedding = encode_with(model, batch)
                atomic_save_tensors({"embedding": embedding}, cache / "pages" / f"{job['sha256']}.safetensors")
                feature_file.unlink()  # embedding first, then its features
                rows.append({"pass": "language", "key": job["sha256"], "doc_id": job["doc_id"], "page": job["page"],
                             "seconds": round(time.perf_counter() - t0, 3), "tokens": int(embedding.shape[0]), "rss_gb": round(process.memory_info().rss / 1e9, 2)})
            stub.features = None
            for question in qtodo:
                t0 = time.perf_counter()
                embedding = encode_with(model, processor(text=[question], return_tensors="pt").to(DEVICE))
                atomic_save_tensors({"embedding": embedding}, cache / "queries" / f"{sha256_bytes(question.encode())}.safetensors")
                rows.append({"pass": "query", "key": sha256_bytes(question.encode()), "doc_id": "", "page": 0,
                             "seconds": round(time.perf_counter() - t0, 3), "tokens": int(embedding.shape[0]), "rss_gb": round(process.memory_info().rss / 1e9, 2)})
        del model, stub
        gc.collect()
        torch.mps.empty_cache()
        log_rows(rows)
        return len(todo), len(qtodo)

    print("Two-pass encoding functions defined (pass 1: vision tower; pass 2: language model with cached vision features).")

In [7]:
# Parity gate (frozen spec): two development pages and one development question, encoded from scratch, must equal E5 bit for bit.
PARITY_FILE = FINAL_DIR / "parity_gate.json"
if MODE in ENCODE_MODES:
    parity_pages = e5_manifest.groupby("doc_id").head(1).sort_values("doc_id").head(2)
    parity_jobs = [{"doc_id": r.doc_id, "page": int(r.page), "sha256": r.sha256, "image_path": r.image_path} for r in parity_pages.itertuples()]
    e8_first = json.loads((PROJECT_ROOT / "experiments" / "frozen" / "e8" / "run_20261125T140433934010Z_4edb90e7_full" / "execution_log.jsonl").open().readline())
    parity_question = e8_first["request"]["messages"][1]["content"][0]["text"].removeprefix("Question: ").removesuffix("\n\nPage images:")
    parity_cache = FINAL_CACHE / "parity"
    for sub in ("pages", "queries", "vision_features"):
        (parity_cache / sub).mkdir(parents=True, exist_ok=True)
    pass1(parity_jobs, parity_cache)
    pass2(parity_jobs, [parity_question], parity_cache)
    checks = {}
    for job in parity_jobs:
        new = load_file(str(parity_cache / "pages" / f"{job['sha256']}.safetensors"))["embedding"]
        old = load_file(str(E5_CACHE_DIR / "pages" / f"{job['sha256']}.safetensors"))["embedding"]
        checks[f"page {job['doc_id'][:30]} p.{job['page']}"] = bool(new.dtype == old.dtype and new.shape == old.shape and torch.equal(new, old))
    qkey = sha256_bytes(parity_question.encode())
    new_q, old_q = (load_file(str(c / "queries" / f"{qkey}.safetensors"))["embedding"] for c in (parity_cache, E5_CACHE_DIR))
    checks["question (E8's first)"] = bool(new_q.shape == old_q.shape and torch.equal(new_q, old_q))
    atomic_write_bytes(PARITY_FILE, json.dumps({"passed": all(checks.values()), "checks": checks, "encoder": ENCODER_CONFIG}, indent=1).encode())
    display(pd.Series(checks, name="bitwise equal to E5").to_frame())
    assert all(checks.values()), "PARITY GATE FAILED: the encoder does not reproduce E5; no pool page may be encoded"
    print("Parity gate passed: fresh encodings equal the frozen E5 cache bit for bit.")
elif PARITY_FILE.is_file():
    print("Parity gate (recorded):", json.loads(PARITY_FILE.read_text())["checks"])

In [8]:
# Encode every pool page (index mode only), 200 pages per chunk: pass 1 then pass 2, so transient features stay below ~0.4 GB.
if MODE == "index":
    assert json.loads(PARITY_FILE.read_text())["passed"], "parity gate not passed"
    assert pool_pages is not None and pool_pages["doc_id"].nunique() == len(POOL_DOCS), "render and text extraction are incomplete"
    jobs = pool_pages.drop_duplicates("sha256")[["doc_id", "page", "sha256", "image_path"]].to_dict("records")
    pending = [j for j in jobs if not (FINAL_CACHE / "pages" / f"{j['sha256']}.safetensors").is_file()]
    print(f"{len(jobs)} distinct page images, {len(pending)} to encode.", flush=True)
    size, started = SETTINGS["encode_chunk_pages"], time.perf_counter()
    for start in range(0, len(pending), size):
        chunk = pending[start:start + size]
        pass1(chunk, FINAL_CACHE)
        pass2(chunk, [], FINAL_CACHE)
        done = start + len(chunk)
        rate = (time.perf_counter() - started) / done
        print(f"  encoded {done}/{len(pending)} pages, {rate:.1f} s/page, about {rate * (len(pending) - done) / 3600:.1f} h left", flush=True)
    assert not any((FINAL_CACHE / "vision_features").glob("*.safetensors")), "unconsumed pass-1 features remain"

if pool_pages is not None and FINAL_CACHE.is_dir():
    encoded = pool_pages["sha256"].map(lambda s: (FINAL_CACHE / "pages" / f"{s}.safetensors").is_file())
    print(f"Pool pages with an embedding: {int(encoded.sum())}/{len(pool_pages)}.")

In [9]:
# The index manifest: every pool page's image hash and embedding hash. It is committed BEFORE any question is loaded.
MANIFEST = FINAL_DIR / "index_manifest.csv"
FROZEN_INDEX = PROJECT_ROOT / "experiments" / "frozen" / "final_index"
if pool_pages is not None and FINAL_CACHE.is_dir():
    complete = pool_pages["doc_id"].nunique() == len(POOL_DOCS) and \
        pool_pages["sha256"].map(lambda s: (FINAL_CACHE / "pages" / f"{s}.safetensors").is_file()).all()
    if complete and not MANIFEST.is_file():
        manifest = pool_pages[["doc_id", "page", "native_words", "ocr_used", "width_px", "height_px", "sha256"]].copy()
        manifest["embedding_sha256"] = manifest["sha256"].map(lambda s: sha256_file(FINAL_CACHE / "pages" / f"{s}.safetensors"))
        manifest["text_sha256"] = [sha256_bytes(t.encode()) for t in pool_pages["text"]]
        atomic_write_bytes(MANIFEST, manifest.to_csv(index=False).encode())
        print(f"Index manifest written: {len(manifest)} pages. Next: freeze it in experiments/frozen/final_index and commit.")
INDEX_FROZEN = (FROZEN_INDEX / "index_manifest.csv").is_file() and not git("status", "--porcelain", "--", str(FROZEN_INDEX.relative_to(PROJECT_ROOT))).strip() \
    and bool(git("ls-files", "--", str((FROZEN_INDEX / "index_manifest.csv").relative_to(PROJECT_ROOT))).strip())
print(f"Index frozen and committed: {INDEX_FROZEN}. Questions {'may now be loaded' if INDEX_FROZEN else 'stay unread'}.")

Index frozen and committed: False. Questions stay unread.


## Stage 2: questions, retrieval, requests and the run

**Arms** (frozen spec):

| Arm | Pages the reader sees |
|---|---|
| **B1** | ColQwen2 MaxSim top 4 |
| **T1** | BM25 top 4 over native text with an OCR fallback |
| **A1** | the ColQwen2 top 8 when the frozen U3 trigger fires, otherwise B1's reply (derived, no extra call when it does not fire) |

The reader is identical everywhere: `gpt-6-luna`, contract b1, temperature 0, reasoning effort none, 800 output tokens. Only the pages differ.

U3 for a document with N pages is $-(s_1 - s_k)/s_1$ with $k = \min(8, N)$. It fires at ≥ −0.153079, and never when N ≤ 4 (B1 already reads every page).

**Every step is first proven on development data**, where frozen answers exist:
- **V:** MaxSim over the E5 cache reproduces E5's frozen ColQwen2 rankings.
- **A:** BM25 over E5's page texts reproduces E5's frozen BM25 rankings.
- **Requests:** built from the raw question text, they reproduce all 118 frozen E8 L-B1 request fingerprints.

Only then are pool requests built.

**Execution.** One session in a seeded order (seed 20261201), with each question's calls in random order. The executor is the fail-closed E8/E9 one:
- an append-only log;
- no SDK retries, and at most 2 attempts per call;
- it stops on an API error, a contract failure, or a change of served identity;
- a cumulative ceiling of $2.50, smoke included. Unknown usage is charged at its upper bound.

The smoke runs on **development** questions only. The plan's totals are frozen in `experiments/specs/final_evaluation_v1_plan.json` before the full run.

In [10]:
# Questions: read only once the index manifest is frozen and committed (spec order). Pool rows and the 118 development rows.
QUESTIONS_PATH = PROJECT_ROOT / "data" / "processed" / "mmlongbench_doc" / "questions.parquet"
QUESTIONS_READY = bool(INDEX_FROZEN)
if QUESTIONS_READY:
    assert QUESTIONS_PATH.is_file(), "run 00_document_processing to build questions.parquet"
    _all_questions = pd.read_parquet(QUESTIONS_PATH)
    pool_q = _all_questions[_all_questions["doc_id"].isin(POOL_DOCS)].set_index("qid").copy()
    dev_q = _all_questions[_all_questions["doc_id"].isin(HELDOUT14)].set_index("qid").copy()
    del _all_questions  # every other row (development or touched) is dropped unread
    for frame in (pool_q, dev_q):
        frame["evidence_pages"] = frame["evidence_pages"].map(lambda pages: [int(p) for p in pages])
        frame["evidence_sources"] = frame["evidence_sources"].map(list)
    assert len(dev_q) == 118 and set(pool_q["doc_id"]) <= set(POOL_DOCS)
    print(f"Pool: {len(pool_q)} questions in {pool_q['doc_id'].nunique()} documents "
          f"({int(pool_q['answerable'].sum())} answerable, {int((~pool_q['answerable']).sum())} unanswerable). "
          f"Development questions for the parity checks and the smoke: {len(dev_q)}.")
else:
    pool_q = dev_q = None
    print("Questions unread: the index manifest is not frozen and committed yet.")

Questions unread: the index manifest is not frozen and committed yet.


In [11]:
# Encode the pool questions (queries mode). The development queries are already in the E5 cache.
def query_file(cache, question):
    return cache / "queries" / f"{sha256_bytes(question.encode())}.safetensors"


if MODE == "queries":
    assert QUESTIONS_READY
    pass2([], sorted(set(pool_q["question"])), FINAL_CACHE)
QUERIES_READY = QUESTIONS_READY and all(query_file(FINAL_CACHE, q).is_file() for q in set(pool_q["question"]))
print(f"Pool question embeddings complete: {QUERIES_READY}.")

Pool question embeddings complete: False.


In [12]:
# Retrieval for both arms, proven on development data first, then on the pool; U3 for A1.
import math
import re
from collections import Counter

TOKEN_PATTERN = re.compile(r"[a-z0-9]+")


def tokenize(text):
    return TOKEN_PATTERN.findall(str(text).lower())


def bm25_rank(page_texts, query, k1=SETTINGS["bm25_k1"], b=SETTINGS["bm25_b"]):
    """BM25 over one PDF ({page: text}); every page, best first, ties by page number (notebook 01 / E5 arm A)."""
    tokens = {page: tokenize(text) for page, text in page_texts.items()}
    counts = {page: Counter(t) for page, t in tokens.items()}
    lengths = {page: len(t) for page, t in tokens.items()}
    avgdl = float(np.mean(list(lengths.values())))
    frequency = Counter(token for t in tokens.values() for token in set(t))
    idf = {token: math.log(1 + (len(tokens) - n + 0.5) / (n + 0.5)) for token, n in frequency.items()}
    terms = set(tokenize(query))
    scored = []
    for page, c in counts.items():
        length_factor = k1 * (1 - b + b * lengths[page] / max(avgdl, 1.0))
        scored.append((page, sum(idf[t] * c[t] * (k1 + 1) / (c[t] + length_factor) for t in terms if c[t])))
    return sorted(scored, key=lambda item: (-item[1], item[0]))


RANKINGS_READY = False
if QUERIES_READY:
    import torch
    from safetensors.torch import load_file

    def maxsim_rank(query, page_embeddings):
        """MaxSim (notebook 04): sum over query vectors of the best page-vector dot product, float32; ties by page number."""
        scored = [(page, float((query @ emb.T).max(dim=1).values.sum())) for page, emb in page_embeddings.items()]
        return sorted(scored, key=lambda item: (-item[1], item[0]))

    def load_embedding(path):
        return load_file(str(path))["embedding"].float()

    # 1. Development parity: E5's frozen rankings (arm V from the E5 cache, arm A from E5's page texts), every page and position.
    frozen_rankings = pd.read_parquet(E5_FROZEN / "tables" / "rankings.parquet").query("population == 'held-out'").set_index(["qid", "arm"])
    heldout_texts = pd.read_parquet(PROJECT_ROOT / "data" / "processed" / "mmlongbench_doc" / "e5" / "heldout_pages.parquet")
    dev_shas = e5_manifest.set_index(["doc_id", "page"])["sha256"]
    worst = 0.0
    for doc_id, group in dev_q.groupby("doc_id"):
        pages = sorted(int(p) for p in e5_manifest.loc[e5_manifest["doc_id"] == doc_id, "page"])
        embeddings = {p: load_embedding(E5_CACHE_DIR / "pages" / f"{dev_shas[(doc_id, p)]}.safetensors") for p in pages}
        texts = dict(zip(heldout_texts.loc[heldout_texts["doc_id"] == doc_id, "page"], heldout_texts.loc[heldout_texts["doc_id"] == doc_id, "text"]))
        for qid, row in group.iterrows():
            v = maxsim_rank(load_embedding(query_file(E5_CACHE_DIR, row["question"])), embeddings)
            assert [p for p, _ in v] == list(frozen_rankings.loc[(qid, "V"), "pages"]), f"{qid}: V ranking differs from E5"
            worst = max(worst, float(np.max(np.abs(np.array([s for _, s in v]) - np.array(frozen_rankings.loc[(qid, "V"), "scores"])))))
            a = bm25_rank(texts, row["question"])
            assert [p for p, _ in a] == list(frozen_rankings.loc[(qid, "A"), "pages"]), f"{qid}: BM25 ranking differs from E5"
    assert worst < 1e-4
    print(f"Development parity: 118/118 ColQwen2 and BM25 rankings equal E5's frozen rankings at every position (max score difference {worst:.1e}).")

    # 2. The pool.
    pool_shas = pool_pages.set_index(["doc_id", "page"])["sha256"]
    rows = []
    for doc_id, group in pool_q.groupby("doc_id"):
        doc_pages = pool_pages[pool_pages["doc_id"] == doc_id]
        embeddings = {int(p): load_embedding(FINAL_CACHE / "pages" / f"{s}.safetensors") for p, s in zip(doc_pages["page"], doc_pages["sha256"])}
        texts = dict(zip(doc_pages["page"].astype(int), doc_pages["text"]))
        for qid, row in group.iterrows():
            v = maxsim_rank(load_embedding(query_file(FINAL_CACHE, row["question"])), embeddings)
            a = bm25_rank(texts, row["question"])
            rows.append({"qid": qid, "doc_id": doc_id, "n_pages": len(embeddings), "V_pages": [p for p, _ in v], "V_scores": [s for _, s in v],
                         "A_pages": [p for p, _ in a], "A_scores": [s for _, s in a]})
        del embeddings
    pool_rankings = pd.DataFrame(rows).set_index("qid")
    U3_THRESHOLD = -0.153079
    k = pool_rankings["n_pages"].clip(upper=8)
    pool_rankings["U3"] = [-(s[0] - s[kk - 1]) / s[0] for s, kk in zip(pool_rankings["V_scores"], k)]
    pool_rankings["triggered"] = (pool_rankings["n_pages"] > 4) & (pool_rankings["U3"] >= U3_THRESHOLD)
    pool_rankings.to_parquet(FINAL_DIR / "rankings.parquet")
    RANKINGS_READY = True
    print(f"Pool rankings: {len(pool_rankings)} questions; U3 triggers on {int(pool_rankings['triggered'].sum())} "
          f"({pool_rankings['triggered'].mean():.1%}; development: 47/118 = 39.8%).")
else:
    print("Retrieval waits for the question embeddings.")

Retrieval waits for the question embeddings.


In [13]:
# Requests, rebuilt visibly from the frozen E8 B1 template and proven on the 118 development questions; then the pool requests.
import base64
from math import ceil, floor

import tiktoken

E8_RUN = PROJECT_ROOT / "experiments" / "frozen" / "e8" / "run_20261125T140433934010Z_4edb90e7_full"
MODEL = "gpt-6-luna"
PRICE = {"input": 0.10 / 1e6, "cached_input": 0.01 / 1e6, "output": 0.50 / 1e6}
NOT_ANSWERABLE = "Not answerable"
ENCODING = tiktoken.get_encoding("o200k_base")


def read_jsonl(path):
    return [json.loads(line) for line in Path(path).read_text().splitlines() if line.strip()]


def png_size(png):
    assert png.startswith(b"\x89PNG\r\n\x1a\n") and len(png) >= 24
    return int.from_bytes(png[16:20], "big"), int.from_bytes(png[20:24], "big")


def loggable(request):
    messages = []
    for message in request["messages"]:
        if isinstance(message["content"], str):
            messages.append(dict(message))
            continue
        parts = []
        for part in message["content"]:
            if part["type"] == "image_url":
                png = base64.b64decode(part["image_url"]["url"].split(",", 1)[1])
                parts.append({"type": "image", "sha256": sha256_bytes(png), "bytes": len(png),
                              "dimensions": list(png_size(png)), "detail": part["image_url"].get("detail")})
            else:
                parts.append(part)
        messages.append({"role": message["role"], "content": parts})
    return {**{k: v for k, v in request.items() if k != "messages"}, "messages": messages}


def fingerprint(request):
    return sha256_bytes(json.dumps(loggable(request), sort_keys=True).encode())


def prompt_tokens(logged):
    texts, dims = [], []
    for message in logged["messages"]:
        for part in ([{"type": "text", "text": message["content"]}] if isinstance(message["content"], str) else message["content"]):
            if part["type"] == "text":
                texts.append(part["text"])
            elif part["type"] == "image":
                dims.append(tuple(part["dimensions"]))
    framing = 9 if dims else 10
    schema = 100 if logged["response_format"]["type"] == "json_schema" else 0
    return sum(len(ENCODING.encode(t)) for t in texts) + framing + schema + sum(floor(1.2 * ceil(w / 32) * ceil(h / 32)) for w, h in dims)


def upper_cost(logged):
    return prompt_tokens(logged) * PRICE["input"] + logged["max_completion_tokens"] * PRICE["output"]


e8_log = read_jsonl(E8_RUN / "execution_log.jsonl")
e8_b1 = {r["qid"]: r for r in e8_log if r["arm"] == "L-B1"}
TEMPLATE = e8_b1[sorted(e8_b1)[0]]["request"]  # system prompt, response_format and parameters of B1, byte for byte
assert all({k: v for k, v in r["request"].items() if k != "messages"} == {k: v for k, v in TEMPLATE.items() if k != "messages"}
           and r["request"]["messages"][0] == TEMPLATE["messages"][0] for r in e8_b1.values())
assert all(r["prompt_tokens"] == prompt_tokens(r["request"]) for r in e8_log if r["arm"] in ("L-B1", "L-B1r"))

IMAGE_INDEX = {}  # (doc_id, page) -> (relative path, sha256): development from E5's manifest, the pool from its page table
IMAGE_INDEX.update({(r.doc_id, int(r.page)): (r.image_path, r.sha256) for r in e5_manifest.itertuples()})
if pool_pages is not None:
    IMAGE_INDEX.update({(r.doc_id, int(r.page)): (r.image_path, r.sha256) for r in pool_pages.itertuples()})


def page_png(doc_id, page):
    relative, digest = IMAGE_INDEX[(doc_id, int(page))]
    png = (PROJECT_ROOT / relative).read_bytes()
    assert sha256_bytes(png) == digest, f"stale render: {doc_id} p.{page}"
    return png


def build_request(question, doc_id, pages):
    """B1's layout: system prompt; 'Question: <q>\n\nPage images:'; then '[Page N]' and the page image, best-ranked first."""
    content = [{"type": "text", "text": f"Question: {question}\n\nPage images:"}]
    for page in pages:
        content.append({"type": "text", "text": f"[Page {page}]"})
        content.append({"type": "image_url", "image_url": {"url": "data:image/png;base64," + base64.b64encode(page_png(doc_id, page)).decode("ascii"),
                                                           "detail": "high"}})
    return {**{k: v for k, v in TEMPLATE.items() if k != "messages"},
            "messages": [dict(TEMPLATE["messages"][0]), {"role": "user", "content": content}]}


REQUESTS_READY = False
if QUESTIONS_READY:
    # Parity: all 118 development B1 requests, built from the raw question text, equal the frozen E8 L-B1 requests.
    for qid, row in dev_q.iterrows():
        request = build_request(row["question"], row["doc_id"], list(frozen_rankings.loc[(qid, "V"), "pages"])[:4]) if RANKINGS_READY else None
        if request is not None:
            assert fingerprint(request) == e8_b1[qid]["fingerprint"] and loggable(request) == e8_b1[qid]["request"], qid
    if RANKINGS_READY:
        print("Request parity: 118/118 development B1 requests built from the question text equal the frozen E8 L-B1 requests.")

if RANKINGS_READY:
    REQUESTS = {}
    for qid, row in pool_q.iterrows():
        r = pool_rankings.loc[qid]
        arms = {"B1": r["V_pages"][:4], "T1": r["A_pages"][:4]}
        if r["triggered"]:
            arms["F8"] = r["V_pages"][:8]
        for arm, pages in arms.items():
            logged = loggable(build_request(row["question"], row["doc_id"], pages))
            REQUESTS[(qid, arm)] = {"doc_id": row["doc_id"], "pages": [int(p) for p in pages], "question": row["question"],
                                    "fingerprint": sha256_bytes(json.dumps(logged, sort_keys=True).encode()),
                                    "prompt_tokens": prompt_tokens(logged), "upper_usd": upper_cost(logged)}
            assert REQUESTS[(qid, arm)]["prompt_tokens"] + 800 <= 128_000
    REQUESTS_READY = True
    print(f"Pool requests built: {len(REQUESTS)} ({Counter(a for _, a in REQUESTS).most_common()}).")


def rebuild(qid, arm, source="pool"):
    """The full request for one planned call, refused unless it equals the planned fingerprint."""
    spec = REQUESTS[(qid, arm)] if source == "pool" else SMOKE_REQUESTS[(qid, arm)]
    request = build_request(spec["question"], spec["doc_id"], spec["pages"])
    assert fingerprint(request) == spec["fingerprint"], f"{qid} {arm}: request changed after planning"
    return request

In [14]:
# The smoke plan (6 development calls) and the full plan (seeded order), with their bounds against the frozen ceilings.
BUDGET_USD, SMOKE_BUDGET_USD, PLAN_UPPER_MAX_USD = 2.50, 0.02, 2.40
PLAN_SPEC_PATH = PROJECT_ROOT / "experiments" / "specs" / "final_evaluation_v1_plan.json"
PLAN_READY = PLAN_FROZEN = False
if REQUESTS_READY:
    # Smoke: two B1, two T1 and two 8-page requests on development questions (never scored).
    smoke_rng = np.random.default_rng(20261204)
    smoke_qids = sorted(smoke_rng.choice(sorted(dev_q.index), size=6, replace=False).tolist())
    SMOKE_REQUESTS = {}
    for qid, arm in zip(smoke_qids, ["B1", "B1", "T1", "T1", "F8", "F8"]):
        ranked = list(frozen_rankings.loc[(qid, "A" if arm == "T1" else "V"), "pages"])
        pages = ranked[:8] if arm == "F8" else ranked[:4]
        logged = loggable(build_request(dev_q.at[qid, "question"], dev_q.at[qid, "doc_id"], pages))
        SMOKE_REQUESTS[(qid, arm)] = {"doc_id": dev_q.at[qid, "doc_id"], "pages": [int(p) for p in pages], "question": dev_q.at[qid, "question"],
                                      "fingerprint": sha256_bytes(json.dumps(logged, sort_keys=True).encode()), "prompt_tokens": prompt_tokens(logged)}
    SMOKE_PLAN = [{"execution_index": i + 1, "replicate_id": "smoke", "qid": qid, "arm": arm, "source": "development",
                   "fingerprint": spec["fingerprint"], "prompt_tokens": spec["prompt_tokens"], "max_completion_tokens": 800}
                  for i, ((qid, arm), spec) in enumerate(SMOKE_REQUESTS.items())]

    order_rng = np.random.default_rng(20261201)
    qids = sorted(pool_q.index)
    order_rng.shuffle(qids)
    FULL_PLAN = []
    for qid in qids:
        arms = [arm for arm in ("B1", "T1", "F8") if (qid, arm) in REQUESTS]
        for arm in order_rng.permutation(arms).tolist():
            spec = REQUESTS[(qid, arm)]
            FULL_PLAN.append({"execution_index": len(FULL_PLAN) + 1, "replicate_id": "final_r1", "qid": qid, "arm": arm, "source": "pool",
                              "fingerprint": spec["fingerprint"], "prompt_tokens": spec["prompt_tokens"], "max_completion_tokens": 800})

    def totals(plan):
        prompt, output = sum(i["prompt_tokens"] for i in plan), sum(i["max_completion_tokens"] for i in plan)
        return {"calls": len(plan), "prompt_tokens": prompt, "max_output_tokens": output,
                "upper_usd": round(prompt * PRICE["input"] + output * PRICE["output"], 6),
                "plan_sha256": sha256_bytes(json.dumps(plan, sort_keys=True).encode())}

    PLAN_DESIGN = {"spec_commit": SPEC_FREEZE_COMMIT, "index_manifest_sha256": sha256_file(FROZEN_INDEX / "index_manifest.csv"),
                   "seeds": {"order": 20261201, "smoke": 20261204}, "smoke": {**totals(SMOKE_PLAN), "qids": smoke_qids},
                   "full": {**totals(FULL_PLAN), "arms": dict(Counter(i["arm"] for i in FULL_PLAN)),
                            "questions": len(pool_q), "triggered": int(pool_rankings["triggered"].sum())},
                   "ceilings": {"cumulative_usd": BUDGET_USD, "smoke_usd": SMOKE_BUDGET_USD, "plan_upper_max_usd": PLAN_UPPER_MAX_USD}}
    assert PLAN_DESIGN["smoke"]["upper_usd"] <= SMOKE_BUDGET_USD
    assert PLAN_DESIGN["full"]["upper_usd"] <= PLAN_UPPER_MAX_USD, "the plan's upper bound exceeds the frozen maximum: stop and ask the owner"
    atomic_write_bytes(FINAL_DIR / "plan_design.json", json.dumps({**PLAN_DESIGN, "smoke_plan": SMOKE_PLAN, "full_plan": FULL_PLAN}, indent=1).encode())
    PLAN_READY = True
    PLAN_FROZEN = PLAN_SPEC_PATH.is_file() and json.loads(PLAN_SPEC_PATH.read_text()) == json.loads(json.dumps(PLAN_DESIGN))
    display(pd.DataFrame({"smoke": PLAN_DESIGN["smoke"], "full": PLAN_DESIGN["full"]}).drop(["plan_sha256", "qids"], errors="ignore"))
    print(f"Plan frozen in {PLAN_SPEC_PATH.relative_to(PROJECT_ROOT)}: {PLAN_FROZEN}.")
else:
    print("The plan waits for the requests.")

The plan waits for the requests.


In [15]:
# The fail-closed executor (as E8/E9) and the B1 reply parser, re-checked on all 236 frozen E8 B1 replies; mock-tested.
from collections import Counter
from datetime import datetime, timezone
from types import SimpleNamespace
import tempfile
import uuid


def page_number(value):
    if isinstance(value, int) and not isinstance(value, bool):
        number = value
    elif isinstance(value, float) and value.is_integer():
        number = int(value)
    elif isinstance(value, str) and value.isascii() and value.isdigit():
        number = int(value)
    else:
        raise ValueError(f"cited page {value!r} is not a page number")
    if number < 1:
        raise ValueError(f"cited page {value!r} is not a page number (pages start at 1)")
    return number


def parse_b1_reply(raw):
    try:
        payload = json.loads(raw)
        if not isinstance(payload, dict) or set(payload) != {"answer", "cited_pages", "evidence"}:
            raise TypeError("B1 output must contain exactly answer, cited_pages and evidence")
        answer = payload["answer"]
        scalar = lambda v: isinstance(v, (str, int, float)) and not isinstance(v, bool)
        if not (scalar(answer) or (isinstance(answer, list) and all(scalar(x) for x in answer))):
            raise TypeError("answer is not a string, number, or list of strings/numbers")
        if not isinstance(payload["cited_pages"], list) or not isinstance(payload["evidence"], list):
            raise TypeError("cited_pages and evidence must be lists")
        cited = [page_number(p) for p in payload["cited_pages"]]
        evidence = []
        for item in payload["evidence"]:
            if not isinstance(item, dict) or set(item) != {"page", "quote"} or not isinstance(item["quote"], str):
                raise TypeError("each evidence item must contain exactly page and a string quote")
            evidence.append({"page": page_number(item["page"]), "quote": item["quote"]})
        if isinstance(answer, str) and answer.strip().lower() == "not answerable":
            answer = NOT_ANSWERABLE
        if answer == NOT_ANSWERABLE and (cited or evidence):
            raise ValueError("Not answerable must have empty cited_pages and evidence")
    except (json.JSONDecodeError, KeyError, TypeError, ValueError) as error:
        return "", [], [], f"unparseable model output: {error}"
    return answer, cited, evidence, None


for record in e8_log:
    if record["arm"] in ("L-B1", "L-B1r"):
        answer, cited, evidence, error = parse_b1_reply(record["raw_response"])
        assert error == record["parse_error"] and record["parsed"] == {"answer": answer, "cited_pages": cited, "evidence": evidence}


def utc_now():
    return datetime.now(timezone.utc).isoformat(timespec="milliseconds")


def observed_cost(record):
    if record.get("completion_tokens") is None:
        return 0.0
    prompt, cached = record.get("prompt_tokens") or 0, record.get("cached_prompt_tokens") or 0
    return max(0, prompt - cached) * PRICE["input"] + cached * PRICE["cached_input"] + record["completion_tokens"] * PRICE["output"]


def budget_charge(record):
    return observed_cost(record) if record.get("completion_tokens") is not None else record.get("cost_upper_usd", 0.0)


def load_log(path, plan):
    records = read_jsonl(path) if Path(path).is_file() else []
    by_index = {item["execution_index"]: item for item in plan}
    successful, attempts = {}, Counter()
    for line, record in enumerate(records, start=1):
        index = record.get("execution_index")
        assert index in by_index and record.get("fingerprint") == by_index[index]["fingerprint"], f"log line {line} is not in the plan"
        attempts[index] += 1
        assert attempts[index] <= 2, f"execution index {index} has more than two attempts"
        if record.get("status") == "ok":
            assert index not in successful
            successful[index] = record
    return records, successful, attempts


def execute(create, run_dir, plan, *, budget_usd, prior_spend=0.0, stop_after=None):
    run_dir = Path(run_dir)
    run_dir.mkdir(parents=True, exist_ok=True)
    log_path = run_dir / "execution_log.jsonl"
    records, successful, attempts = load_log(log_path, plan)
    spent = prior_spend + sum(budget_charge(r) for r in records)
    observed = sum(observed_cost(r) for r in records)
    models = {r.get("served_model") for r in successful.values()}
    fingerprints = {r.get("system_fingerprint") for r in successful.values()}
    done_now = 0
    for item in plan:
        index = item["execution_index"]
        if index in successful:
            continue
        if attempts[index] >= 2:
            raise RuntimeError(f"execution index {index} exhausted its two attempts")
        request = rebuild(item["qid"], item["arm"], "development" if item["source"] == "development" else "pool")
        logged = loggable(request)
        upper = upper_cost(logged)
        if spent + upper > budget_usd:
            raise ValueError(f"the next request could exceed the cumulative ceiling: ${spent + upper:.6f} > ${budget_usd:.6f}")
        record = {**item, "attempt": attempts[index] + 1, "started_at": utc_now(), "status": "started", "cost_upper_usd": upper, "request": logged}
        started = time.perf_counter()
        try:
            response = create(**request)
        except Exception as error:
            record.update(status="api_error", error=f"{type(error).__name__}: {str(error)[:300]}", completed_at=utc_now(),
                          latency_s=round(time.perf_counter() - started, 3))
            with log_path.open("a") as handle:
                handle.write(json.dumps(record) + "\n")
            raise RuntimeError(f"stopped after an API failure at execution index {index}; resume is explicit") from error
        choice = response.choices[0]
        raw = choice.message.content or ""
        answer, cited, evidence, error = parse_b1_reply(raw)
        if choice.finish_reason != "stop" and error is None:
            error = f"reply did not finish normally (finish_reason={choice.finish_reason!r})"
        usage = getattr(response, "usage", None)
        details = getattr(usage, "prompt_tokens_details", None) if usage else None
        record.update(status="ok" if error is None else "contract_error", completed_at=utc_now(), latency_s=round(time.perf_counter() - started, 3),
                      served_model=getattr(response, "model", None), system_fingerprint=getattr(response, "system_fingerprint", None),
                      finish_reason=choice.finish_reason, raw_response=raw, parsed={"answer": answer, "cited_pages": cited, "evidence": evidence},
                      parse_error=error, prompt_tokens=getattr(usage, "prompt_tokens", None), completion_tokens=getattr(usage, "completion_tokens", None),
                      cached_prompt_tokens=getattr(details, "cached_tokens", 0) if details else 0)
        record["list_price_cost_usd"] = observed_cost(record)
        new_models, new_fingerprints = models | {record["served_model"]}, fingerprints | {record["system_fingerprint"]}
        if len(new_models) != 1 or len(new_fingerprints) != 1:
            record["status"] = "identity_mismatch"
        with log_path.open("a") as handle:
            handle.write(json.dumps(record) + "\n")
        records.append(record)
        attempts[index] += 1
        spent += budget_charge(record)
        observed += record["list_price_cost_usd"]
        if record["status"] == "identity_mismatch":
            raise RuntimeError(f"served model or system fingerprint changed: {sorted(map(str, new_models))}, {sorted(map(str, new_fingerprints))}")
        if error is not None:
            raise RuntimeError(f"stopped on a contract failure at execution index {index}: {error}")
        models, fingerprints = new_models, new_fingerprints
        successful[index] = record
        done_now += 1
        if stop_after is not None and done_now >= stop_after:
            break
    summary = {"complete": len(successful) == len(plan), "successful": len(successful), "planned": len(plan), "attempts": len(records),
               "observed_list_price_cost_usd": round(observed, 9), "budget_accounted_usd_including_prior": round(spent, 9),
               "served_models": sorted(map(str, models)), "system_fingerprints": sorted(map(str, fingerprints)), "updated_at": utc_now()}
    (run_dir / "summary.json").write_text(json.dumps(summary, indent=2) + "\n")
    return summary


class MockCreate:
    """Interface test double (never a project result): a valid abstention with exact usage."""
    def __init__(self, *, fail_on=None, malformed_on=None, model_change_on=None):
        self.calls, self.fail_on, self.malformed_on, self.model_change_on = 0, fail_on, malformed_on, model_change_on

    def __call__(self, **request):
        self.calls += 1
        if self.calls == self.fail_on:
            raise TimeoutError("mock transport interruption")
        raw = "{}" if self.calls == self.malformed_on else json.dumps({"answer": NOT_ANSWERABLE, "cited_pages": [], "evidence": []})
        usage = SimpleNamespace(prompt_tokens=prompt_tokens(loggable(request)), completion_tokens=12, prompt_tokens_details=SimpleNamespace(cached_tokens=0))
        return SimpleNamespace(choices=[SimpleNamespace(message=SimpleNamespace(content=raw), finish_reason="stop")],
                               model=MODEL + ("-changed" if self.calls == self.model_change_on else ""), system_fingerprint=None, usage=usage)


if PLAN_READY:
    mock_plan = SMOKE_PLAN
    with tempfile.TemporaryDirectory() as tmp:
        mock = MockCreate()
        assert not execute(mock, Path(tmp) / "a", mock_plan, budget_usd=BUDGET_USD, stop_after=2)["complete"]
        assert execute(mock, Path(tmp) / "a", mock_plan, budget_usd=BUDGET_USD)["complete"] and mock.calls == 6
        try:
            execute(MockCreate(fail_on=2), Path(tmp) / "b", mock_plan, budget_usd=BUDGET_USD)
            raise AssertionError
        except RuntimeError as err:
            assert "API failure" in str(err)
        resumed = execute(MockCreate(), Path(tmp) / "b", mock_plan, budget_usd=BUDGET_USD)
        assert resumed["complete"] and resumed["attempts"] == 7
        refused = MockCreate()
        try:
            execute(refused, Path(tmp) / "c", mock_plan, budget_usd=1e-9)
            raise AssertionError
        except ValueError:
            assert refused.calls == 0
        for kwargs, message in (({"malformed_on": 1}, "contract failure"), ({"model_change_on": 2}, "changed")):
            try:
                execute(MockCreate(**kwargs), Path(tmp) / f"d{list(kwargs)[0]}", mock_plan, budget_usd=BUDGET_USD)
                raise AssertionError
            except RuntimeError as err:
                assert message in str(err)
    print("Parser reproduces all 236 frozen E8 B1 parses; executor mock tests pass (resume, API failure, budget, contract, identity).")

In [16]:
# The paid gate. smoke: 6 development calls. full: the frozen plan, after a passed smoke. Nothing is sent in any other mode.
SMOKE_RUNS, FULL_RUNS = PROJECT_ROOT / "experiments" / "runs" / "final_smoke", PROJECT_ROOT / "experiments" / "runs" / "final"


def latest(runs_dir, suffix):
    runs = sorted(p for p in runs_dir.glob(f"*_{suffix}") if (p / "summary.json").is_file())
    return runs[-1] if runs else None


def smoke_gate(run_dir):
    records = read_jsonl(run_dir / "execution_log.jsonl")
    ok = [r for r in records if r["status"] == "ok"]
    gate = {"6/6 replies": len(ok) == 6, "finish_reason stop": all(r["finish_reason"] == "stop" for r in ok),
            "strict schema parsed": all(r["parse_error"] is None for r in ok), "one served model, the alias": {r["served_model"] for r in ok} == {MODEL},
            "billed prompt tokens equal the token model": all(r["prompt_tokens"] == prompt_tokens(r["request"]) for r in ok),
            "smoke cost within $0.02": sum(observed_cost(r) for r in records) <= SMOKE_BUDGET_USD}
    return gate, sum(budget_charge(r) for r in records)


if MODE in {"smoke", "full"}:
    from dotenv import load_dotenv
    load_dotenv(PROJECT_ROOT / ".env")
    smoke_dir = latest(SMOKE_RUNS, "smoke")
    smoke_ok, smoke_spend = smoke_gate(smoke_dir) if smoke_dir else ({}, 0.0)
    committed = lambda paths: subprocess.run(["git", "diff", "--quiet", "HEAD", "--", *paths], cwd=PROJECT_ROOT).returncode == 0 and \
        not git("ls-files", "--others", "--exclude-standard", "--", *paths).strip()
    readiness = {"plan built": PLAN_READY, "plan frozen in the plan spec": PLAN_FROZEN, "index frozen": INDEX_FROZEN,
                 "OPENAI_API_KEY present (value never printed)": bool(os.getenv("OPENAI_API_KEY")),
                 "notebook, spec and plan committed": committed(["notebooks/09_final_evaluation.ipynb", "experiments/specs/final_evaluation_v1.json",
                                                                "experiments/specs/final_evaluation_v1_plan.json"])}
    if MODE == "full":
        readiness["smoke passed every gate"] = bool(smoke_ok) and all(smoke_ok.values())
    display(pd.Series(readiness, name="ready").to_frame())
    assert all(readiness.values()), "paid gate closed: nothing was sent"
    from openai import OpenAI
    client = OpenAI(max_retries=0, timeout=120)
    runs_dir, plan, suffix = (SMOKE_RUNS, SMOKE_PLAN, "smoke") if MODE == "smoke" else (FULL_RUNS, FULL_PLAN, "full")
    requested = os.getenv("VISUALDOC_FINAL_RUN_ID", "").strip()
    if requested:
        run_dir = runs_dir / requested
        assert run_dir.is_dir()
    else:
        run_id = f"{datetime.now(timezone.utc):%Y%m%dT%H%M%S%fZ}_{uuid.uuid4().hex[:8]}_{suffix}"
        run_dir = runs_dir / run_id
        run_dir.mkdir(parents=True, exist_ok=False)
        (run_dir / "settings.json").write_text(json.dumps({"run_id": run_id, "mode": MODE, "git_commit": git("rev-parse", "HEAD").decode().strip(),
            "spec_sha256": sha256_file(SPEC_PATH), "plan_spec_sha256": sha256_file(PLAN_SPEC_PATH), "budget_usd": BUDGET_USD,
            "prior_spend_usd": smoke_spend if MODE == "full" else 0.0, "plan": plan}, indent=1) + "\n")
    settings = json.loads((run_dir / "settings.json").read_text())
    run_summary = execute(client.chat.completions.create, run_dir, plan, budget_usd=SMOKE_BUDGET_USD if MODE == "smoke" else BUDGET_USD,
                          prior_spend=settings["prior_spend_usd"])
    display(pd.Series(run_summary, name=f"final {MODE}").to_frame())
    if MODE == "smoke":
        gate, _ = smoke_gate(run_dir)
        (run_dir / "smoke_gate.json").write_text(json.dumps(gate, indent=2) + "\n")
        display(pd.Series(gate, name="smoke gate").to_frame())
    print(f"Outputs: {run_dir.relative_to(PROJECT_ROOT)}")
else:
    print(f"Paid gate closed (VISUALDOC_FINAL_MODE={MODE}): no client was created and no request was sent.")

Paid gate closed (VISUALDOC_FINAL_MODE=offline): no client was created and no request was sent.


## Stage 3: scoring and the frozen claims

**Scoring.**
- **S-v1 is primary.** It is notebook 03 cell `f653d232`: correct when score > 0, over answerable questions.
- **S-v2 is secondary.** Its gray replies go to an arm-blind queue for the owner. Until labelled, each contrast is reported as its exact range over every consistent labelling.

The scorer cells below are verbatim, SHA-pinned copies of notebook 07's.

**Claims** (spec `claims_and_rules`):
- **C1.** B1's accuracy with a document-cluster 95 % interval, overall and on the core scope.
- **C2 (confirmatory).** Δ = acc(B1) − acc(T1).
  - `visual_retrieval_confirmed` if Δ > 0, the one-sided bootstrap 5th percentile is > 0 and the sign-flip p ≤ 0.05.
  - `reversed` if the 97.5th percentile is < 0.
  - Otherwise `visual_retrieval_not_confirmed`.
- **C3 (secondary).** acc(A1) − acc(B1): `a1_gain_supported` if the 5th percentile is > 0 and p ≤ 0.05. Reported only; B1 stays the shipped system.

**Statistics.**
- Unit: question; cluster: document.
- Bootstrap: 10,000 draws, seed 20261202.
- Sign-flip test: one-sided Monte Carlo, 100,000 draws, seed 20261203, on per-document summed differences, $p = (1 + \#\{T^* \ge T\})/(1 + 100{,}000)$.
- McNemar counts are descriptive.

**Not done here, by rule.** There is no per-question error analysis on the pool, and nothing measured here changes the shipped system.

In [17]:
# Names the verbatim scorer cells expect (as notebook 07's setup provides them).
import ast
import inspect
import re
from math import isclose

NB03_PATH = PROJECT_ROOT / "notebooks" / "03_evaluation.ipynb"
FROZEN_DIR = PROJECT_ROOT / "experiments" / "frozen"
assert NB03_PATH.is_file() and (FROZEN_DIR / "e7" / "design" / "e6_error_audit.csv").is_file()

In [18]:
# S-v1: the frozen MMLongBench-Doc scorer (notebook 03, cell f653d232), written out unchanged. Correct = score > 0; strict = score == 1.
def levenshtein_distance(s1, s2):
    if len(s1) > len(s2):
        s1, s2 = s2, s1
    distances = range(len(s1) + 1)
    for i2, c2 in enumerate(s2):
        new_distances = [i2 + 1]
        for i1, c1 in enumerate(s1):
            new_distances.append(distances[i1] if c1 == c2 else 1 + min(distances[i1], distances[i1 + 1], new_distances[-1]))
        distances = new_distances
    return distances[-1]


def anls_compute(groundtruth, prediction, threshold=0.5):
    length = max(len(groundtruth.upper()), len(prediction.upper()))
    anls = 1.0 - (0.0 if length == 0 else levenshtein_distance(groundtruth, prediction) / length)
    return 0.0 if anls <= threshold else anls


def is_float_equal(reference, prediction, include_percentage=False, is_close=False):
    def get_precision(value):
        text = str(value)
        return len(text.split(".")[-1]) if "." in text else 3

    reference = float(str(reference).strip().rstrip("%").strip())
    try:
        prediction = float(str(prediction).strip().rstrip("%").strip())
    except (TypeError, ValueError):
        return False
    for item in ([reference / 100, reference, reference * 100] if include_percentage else [reference]):
        if is_close and isclose(item, prediction, rel_tol=0.01):
            return True
        precision = max(min(get_precision(prediction), get_precision(item)), 2)
        if round(prediction, precision) == round(item, precision):
            return True
    return False


def get_clean_string(s):
    s = str(s).lower().strip()
    # Official quirk kept: the code calls s.rstrip("mile") etc. without assigning the result, so unit suffixes stay.
    s = re.sub(r"\s*\([^)]*\)", "", s).strip()  # remove parenthesised text
    s = re.sub(r"^['\"]|['\"]$", "", s).strip()  # remove surrounding quotes
    s = s.strip().lstrip("$").strip()
    return s.strip().rstrip("%").strip()


EXACT_MATCH_PATTERNS = [
    r"\b\d+(-\d+|\s\d+)?\b",  # phone-like numbers
    r"\b\d{4}[-\s]\d{2}[-\s]\d{2}\b",  # YYYY-MM-DD
    r"\b\d{4}[-\s]\d{2}\b",  # YYYY-MM
    r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}",  # e-mail address
]


def is_exact_match(s):
    return ("https://" in s or s.endswith(".py") or s.endswith("ipynb") or s.startswith("page") or "a.m." in s or "p.m." in s
            or any(re.fullmatch(pattern, s) for pattern in EXACT_MATCH_PATTERNS))


def isfloat(value):
    try:
        float(value)
        return True
    except (TypeError, ValueError):
        return False


def as_list(value):
    if isinstance(value, str) and value.startswith("["):
        try:
            value = ast.literal_eval(value)
        except (ValueError, SyntaxError):
            pass  # deviation: the official eval() would raise here
    return value if isinstance(value, list) else [value]


def eval_score(gt, pred, answer_type):
    if answer_type == "Int":
        try:
            gt, pred = int(gt), int(float(pred))
        except Exception:  # the official code uses a bare except
            pred = ""
        score = gt == pred
    elif answer_type == "Float":
        try:
            gt, pred = float(get_clean_string(str(gt))), float(get_clean_string(str(pred)))
        except Exception:
            pred = ""
        score = is_float_equal(gt, pred, include_percentage=True, is_close=True)
    elif answer_type in ("Str", "None"):
        gt, pred = get_clean_string(gt), get_clean_string(pred)
        score = gt == pred if is_exact_match(gt) else anls_compute(gt, pred)
    else:  # List
        gt, pred = as_list(gt), as_list(pred)
        if len(gt) != len(pred):
            score = 0.0
        else:
            gt, pred = sorted(get_clean_string(i) for i in gt), sorted(get_clean_string(i) for i in pred)
            score = "-".join(gt) == "-".join(pred) if isfloat(gt[0]) or is_exact_match(gt[0]) else min(anls_compute(g, p) for g, p in zip(gt, pred))
    return float(score)


# 1. Unchanged: every function above has the same source as the frozen cell, and so does the pattern list.
frozen_cell = next("".join(c["source"]) for c in json.loads(NB03_PATH.read_text())["cells"] if c.get("id") == "f653d232")
frozen_tree = ast.parse(frozen_cell)
frozen_functions = {node.name: ast.get_source_segment(frozen_cell, node) for node in frozen_tree.body if isinstance(node, ast.FunctionDef)}
SCORER_FUNCTIONS = ["levenshtein_distance", "anls_compute", "is_float_equal", "get_clean_string", "is_exact_match", "isfloat", "as_list", "eval_score"]
assert set(SCORER_FUNCTIONS) <= set(frozen_functions)
assert all(inspect.getsource(globals()[name]).strip() == frozen_functions[name].strip() for name in SCORER_FUNCTIONS)
frozen_patterns = next(ast.literal_eval(node.value) for node in frozen_tree.body
                       if isinstance(node, ast.Assign) and getattr(node.targets[0], "id", None) == "EXACT_MATCH_PATTERNS")
assert frozen_patterns == EXACT_MATCH_PATTERNS

# 2. The 17 hand-traced cases of notebook 03.
SCORING_CASES = [  # reference, prediction, format, expected
    ("Less well-off", "less well off", "Str", 1 - 1 / 13), ("Pew Research Center", "Pew Research Centre", "Str", 1 - 2 / 19),
    ("Latinos", "Asian Americans", "Str", 0.0), ("Not answerable", "Not answerable", "None", 1.0), ("Not answerable", "The report does not say", "None", 0.0),
    ("12", "12.0", "Int", 1.0), ("12", "twelve", "Int", 0.0), ("18.29%", "18.3", "Float", 1.0), ("0.1", "10%", "Float", 1.0), ("51%", "49", "Float", 0.0),
    ("['Circle', 'Rectangle']", "['rectangle', 'circle']", "List", 1.0), ("['White', '10%']", "['White']", "List", 0.0),
    ("['2014', '2016']", "['2016', '2014']", "List", 1.0), ("https://www.example.com", "www.example.com", "Str", 0.0), ("2016-06", "2016-06", "Str", 1.0),
    ("$1,200 (approx.)", "1,200", "Str", 1.0), ("5 miles", "5", "Str", 0.0)]
assert len(SCORING_CASES) == 17 and all(isclose(eval_score(r, p, f), e, abs_tol=1e-9) for r, p, f, e in SCORING_CASES)
print(f"S-v1: {len(SCORER_FUNCTIONS)} functions and the pattern list equal notebook 03's frozen cell f653d232; {len(SCORING_CASES)} hand-traced cases pass.")

S-v1: 8 functions and the pattern list equal notebook 03's frozen cell f653d232; 17 hand-traced cases pass.


In [19]:
# S-v2: a deterministic layer over S-v1 that returns (class, reason) with class in {correct, wrong, gray}. Gray goes to blind adjudication.
SCALE_WORDS = {"thousand": 1e3, "k": 1e3, "million": 1e6, "millions": 1e6, "mn": 1e6, "billion": 1e9, "billions": 1e9, "bn": 1e9, "trillion": 1e12}
NUMBER_PATTERN = re.compile(r"(\()?(-?\d[\d,]*(?:\.\d+)?)(\))?\s*(%|(?:percent|thousand|millions?|billions?|trillion|bn|mn|k)\b)?")
NUMERIC_TEXT_RESIDUE = re.compile(r"us\$|usd|eur|gbp|percent|thousand|millions?|billions?|trillion|\bbn\b|\bmn\b|\bk\b|[$€£%(),.\-\s\d]")
POWERS_OF_1000 = (1e-9, 1e-6, 1e-3, 1e3, 1e6, 1e9)  # "m" and "b" are not scale words: metres, bytes


def numbers_in(value):
    """Every number in a reply or reference, after removing currency signs; a scale word multiplies, '(x)' is negative."""
    if isinstance(value, bool):
        return []
    if isinstance(value, (int, float)):
        return [float(value)]
    text = re.sub(r"us\$|usd|eur|gbp|[$€£]", " ", str(value).lower())
    found = []
    for open_paren, digits, close_paren, unit in NUMBER_PATTERN.findall(text):
        number = float(digits.replace(",", ""))
        found.append((-number if open_paren and close_paren else number) * SCALE_WORDS.get(unit, 1.0))
    return found


def numeric_text(value):
    """True when a value is one number with at most currency, %, separators and a scale word around it."""
    if isinstance(value, bool):
        return False
    if isinstance(value, (int, float)):
        return True
    return NUMERIC_TEXT_RESIDUE.sub("", str(value).lower()) == "" and len(numbers_in(value)) == 1


def same_value(reference, prediction, answer_format):
    # Float: within 1 %, including the official x100 / /100 percentage variants; Int: exactly equal.
    if answer_format == "Float":
        return any(isclose(candidate, prediction, rel_tol=0.01, abs_tol=1e-9) for candidate in (reference, reference / 100, reference * 100))
    return isclose(reference, prediction, rel_tol=0.0, abs_tol=1e-9)


def numeric_class(reference, prediction, answer_format):
    """correct / scale / among / wrong / None (the reference is not one number)."""
    ref, pred = numbers_in(reference), numbers_in(prediction)
    if len(ref) != 1 or not pred:
        return None if len(ref) != 1 else "wrong"
    if len(pred) > 1:
        return "among" if any(same_value(ref[0], p, answer_format) for p in pred) else "wrong"
    if same_value(ref[0], pred[0], answer_format):
        return "correct"
    if any(same_value(ref[0] * f, pred[0], answer_format) for f in POWERS_OF_1000):
        return "scale"
    return "wrong"


def normalise(value):
    text = json.dumps(value) if isinstance(value, list) else str(value)
    return " ".join(re.sub(r"[^a-z0-9%.]+", " ", text.lower()).split())


def contains_phrase(haystack, needle):
    return bool(needle) and f" {needle} " in f" {haystack} "


def canonical_item(item):
    """A list item for comparison: a number as a number ('$35,000' = 35000), anything else normalised text."""
    return f"{numbers_in(item)[0]:.6g}" if numeric_text(item) else normalise(item)


def item_set(value):
    return {canonical_item(item) for item in as_list(value)} - {""}


def items_match(a, b):
    """Equal, one a phrase of the other, or ANLS above 0.5 (S-v1's threshold)."""
    return a == b or contains_phrase(a, b) or contains_phrase(b, a) or anls_compute(a, b) > 0


# References the frozen E6 error audit labelled as annotation issues: scored by S-v1 as usual, sent to adjudication by S-v2.
e6_audit = pd.read_csv(FROZEN_DIR / "e7" / "design" / "e6_error_audit.csv")
DEBATABLE_QIDS = set(e6_audit.loc[e6_audit["category"] == "annotation issue", "qid"])


NUMERIC_REASON = {"correct": ("correct", "numeric match after normalisation (currency signs, separators, %, scale words)"),
                  "scale": ("gray", "same number up to a factor of 1,000^n (unit scale ambiguous)"),
                  "among": ("gray", "the reference value is one of several numbers in the reply"),
                  "wrong": ("wrong", "numeric mismatch")}


def score_v2(reference, prediction, answer_format, debatable=False):
    s1 = eval_score(reference, prediction, answer_format)
    is_abstention = isinstance(prediction, str) and prediction.strip().lower() == NOT_ANSWERABLE.lower()
    if answer_format == "None":
        return ("correct", "exact abstention") if s1 >= 1 else ("wrong", "answered an unanswerable question")
    if is_abstention:
        return "wrong", "false abstention"
    if s1 >= 1:
        return "correct", "S-v1 full match"
    if debatable:
        return "gray", "reference flagged as debatable by the frozen E6 audit"
    if answer_format in ("Int", "Float"):
        return NUMERIC_REASON[numeric_class(reference, prediction, answer_format) or "wrong"]
    if answer_format == "Str":
        if numeric_text(reference) and numeric_text(prediction):
            numeric = numeric_class(reference, prediction, "Float")
            if numeric in ("correct", "scale"):
                return NUMERIC_REASON[numeric]
        ref, pred = normalise(reference), normalise(prediction)
        if ref and ref == pred:
            return "correct", "equal after case and punctuation normalisation"
        if 0 < s1 < 1:
            return "gray", "partial ANLS credit (0.5 < ANLS < 1)"
        if contains_phrase(pred, ref):
            return "gray", "the reference is a phrase inside a longer reply"
        if len(pred.split()) >= 2 and contains_phrase(ref, pred):
            return "gray", "the reply is a phrase inside the reference"
        return "wrong", "string mismatch"
    # List: a reply missing a reference item is wrong (the benchmark asks for the whole list); only full matches go to a human.
    ref_items, pred_items = item_set(reference), item_set(prediction)
    if ref_items and ref_items == pred_items:
        return "correct", "same items after normalisation"
    if ref_items and all(any(items_match(r, p) for p in pred_items) for r in ref_items):
        if len(pred_items) > len(ref_items):
            return "gray", "every reference item present, with extra items"
        return "gray", "every reference item matched up to formatting"
    return "wrong", "a reference item is missing"


# Hand checks on invented values, including what S-v2 cannot settle (negation, debatable references): those stay gray or wrong for a human.
V2_CASES = [  # reference, reply, format, expected class
    ("Not answerable", NOT_ANSWERABLE, "None", "correct"), ("Not answerable", "7", "None", "wrong"), ("7", NOT_ANSWERABLE, "Int", "wrong"),
    ("1234", "1,234 units", "Int", "correct"), ("12", "12%", "Int", "correct"), ("15", "between 15 and 20", "Int", "gray"),
    ("3.5", "3.5 million", "Float", "gray"), ("1200", "$1.2 billion", "Float", "gray"), ("42.0", "17.3", "Float", "wrong"),
    ("300 million", "300", "Str", "gray"), ("Northern region", "The northern region had the most visitors", "Str", "gray"),
    ("Special Agent Ana Lee", "Ana Lee", "Str", "gray"), ("Ana Lee", "Bo Chen", "Str", "wrong"),
    ("Paris", "Not Paris but Lyon", "Str", "gray"),  # failure case: containment cannot see the negation; the adjudicator must
    ("Less well-off", "less well off", "Str", "correct"), ("Pew Research Center", "Pew Research Centre", "Str", "gray"),
    ("['Circle', 'Rectangle']", ["rectangle", "circle"], "List", "correct"), ("['$35,000', '$40,684']", [35000, 40684], "List", "correct"),
    ("['Apple Inc', 'Banana']", ["apple", "banana co"], "List", "gray"), ("['a', 'b']", ["a", "b", "c"], "List", "gray"),
    ("['a', 'b', 'c']", ["a", "b"], "List", "wrong"), ("['a']", ["x"], "List", "wrong")]
assert score_v2("12", "13", "Int", debatable=True) == ("gray", "reference flagged as debatable by the frozen E6 audit")
assert score_v2("12", "12", "Int", debatable=True)[0] == "correct"  # S-v1 = 1 stays correct
observed = {(r, str(p), f): score_v2(r, p, f)[0] for r, p, f, _ in V2_CASES}
assert all(observed[(r, str(p), f)] == expected for r, p, f, expected in V2_CASES), {k: v for k, v in observed.items()}
print(f"S-v2: {len(V2_CASES)} invented cases classified as expected (including a negation that only a human can resolve).")

S-v2: 22 invented cases classified as expected (including a negation that only a human can resolve).


In [20]:
# Both scorer cells are byte-identical to notebook 07's pinned cells.
this_nb = {c.get("id"): "".join(c["source"]) for c in json.loads((PROJECT_ROOT / "notebooks" / "09_final_evaluation.ipynb").read_text())["cells"]}
nb07_cells = {c.get("id"): "".join(c["source"]) for c in json.loads((PROJECT_ROOT / "notebooks" / "07_baseline_consolidation.ipynb").read_text())["cells"]}
for src, here, pin in (("b1-scorer-v1", "fe-scorer-v1", "329f91f3ceda29146ec8a0c2467f11fdf7fdbd66c1e6af29123e1983a3fc7ee7"),
                       ("b1-scorer-v2", "fe-scorer-v2", "902708ba537d580db8c67858fe9a2231e5b546f1538a2b955a3eadff551b8707")):
    assert sha256_bytes(this_nb[here].encode()) == pin == sha256_bytes(nb07_cells[src].encode()), here
print("Scorer cells equal notebook 07's pinned copies; S-v1 equals notebook 03's frozen cell f653d232.")

Scorer cells equal notebook 07's pinned copies; S-v1 equals notebook 03's frozen cell f653d232.


In [21]:
# The evaluation: reads a completed full run (the frozen package once it exists), never calls an API, applies the frozen rules.
FINAL_FROZEN = PROJECT_ROOT / "experiments" / "frozen" / "final"
REPORT_TYPES = {"Financial report", "Research report / Introduction", "Administration/Industry file"}
requested_eval = os.getenv("VISUALDOC_FINAL_EVAL_RUN", "").strip()
frozen_runs = sorted(FINAL_FROZEN.glob("run_*_full")) if FINAL_FROZEN.is_dir() else []
live_runs = sorted(p.parent for p in FULL_RUNS.glob("*_full/summary.json") if json.loads(p.read_text()).get("complete")) if FULL_RUNS.is_dir() else []
EVAL_RUN = PROJECT_ROOT / requested_eval if requested_eval else (frozen_runs[-1] if frozen_runs else (live_runs[-1] if live_runs else None))
FINAL_EVALUATED = EVAL_RUN is not None and PLAN_READY
if not FINAL_EVALUATED:
    print("No completed final run yet: the evaluation waits for the paid run.")
else:
    records = read_jsonl(EVAL_RUN / "execution_log.jsonl")
    settings = json.loads((EVAL_RUN / "settings.json").read_text())
    assert settings["plan"] == FULL_PLAN, "the run's plan is not this notebook's plan"
    ok = [r for r in records if r["status"] == "ok"]
    assert len(ok) == len(FULL_PLAN) == len({r["execution_index"] for r in ok})
    assert {r["served_model"] for r in ok} == {MODEL} and len({r["system_fingerprint"] for r in ok}) == 1
    assert all(r["finish_reason"] == "stop" and r["parse_error"] is None for r in ok)

    rows = []
    for r in ok:
        qid, arm = r["qid"], r["arm"]
        answer, cited = r["parsed"]["answer"], r["parsed"]["cited_pages"]
        ref, fmt = pool_q.at[qid, "answer"], pool_q.at[qid, "answer_format"]
        s1 = eval_score(ref, answer, fmt)
        v2_class, v2_reason = score_v2(ref, answer, fmt, debatable=qid in DEBATABLE_QIDS)
        supplied = REQUESTS[(qid, arm)]["pages"]
        rows.append({"qid": qid, "doc_id": pool_q.at[qid, "doc_id"], "arm": arm, "answerable": bool(pool_q.at[qid, "answerable"]),
                     "s1_score": s1, "s1_correct": s1 > 0, "v2_class": v2_class, "v2_reason": v2_reason, "prediction": answer,
                     "citation_valid": set(cited) <= set(supplied) and (answer == NOT_ANSWERABLE or bool(cited)), "schema_valid": True,
                     "abstained": answer == NOT_ANSWERABLE, "pages_read": len(supplied), "prompt_tokens": r["prompt_tokens"],
                     "completion_tokens": r["completion_tokens"], "cost_usd": r["list_price_cost_usd"], "latency_s": r["latency_s"]})
    paid = pd.DataFrame(rows)
    by_arm = {arm: frame.set_index("qid") for arm, frame in paid.groupby("arm")}
    a1 = [{**by_arm["F8" if pool_rankings.at[qid, "triggered"] else "B1"].loc[qid].to_dict(), "qid": qid, "arm": "A1"} for qid in pool_q.index]
    scores = pd.concat([paid, pd.DataFrame(a1)], ignore_index=True)

    # S-v2: owner labels only for identical blind items already labelled; everything else is pending (never a model judgment).
    def blind_row(qid, prediction):
        return {"document": pool_q.at[qid, "doc_id"], "evidence_pages": " ".join(map(str, pool_q.at[qid, "evidence_pages"])),
                "question": pool_q.at[qid, "question"], "answer_format": pool_q.at[qid, "answer_format"], "reference": pool_q.at[qid, "answer"],
                "reply": json.dumps(prediction) if isinstance(prediction, list) else str(prediction)}

    def blind_identity(row):
        return sha256_bytes(json.dumps({c: str(row[c]) for c in ["document", "evidence_pages", "question", "answer_format", "reference", "reply"]},
                                       sort_keys=True, ensure_ascii=False).encode())

    owner_labels = {}
    for package in ("scoring_v2", "e8_scoring_v2", "e9_scoring_v2", "final_scoring_v2"):
        folder = FROZEN_DIR / package
        if (folder / "SHA256SUMS").is_file():
            labels = pd.read_csv(folder / "adjudication_labels.csv", dtype=str).fillna("")
            owner_labels.update(dict(zip(labels["identity_sha256"], labels["label"])))
    scores["identity"] = [blind_identity(blind_row(q, p)) for q, p in zip(scores["qid"], scores["prediction"])]
    scores["v2_final"] = np.where(scores["v2_class"] == "gray", scores["identity"].map(owner_labels).fillna("pending"), scores["v2_class"])
    pending = scores[(scores["v2_final"] == "pending") & scores["answerable"] & scores["arm"].isin(["B1", "T1", "F8"])]
    items = pending.drop_duplicates("identity")[["identity", "qid", "prediction"]].sample(frac=1.0, random_state=20261205).reset_index(drop=True)
    items["item_id"] = [f"FINAL-ADJ-{i + 1:04d}" for i in range(len(items))]
    queue = pd.DataFrame([{"item_id": r.item_id, **blind_row(r.qid, r.prediction)} for r in items.itertuples()])
    if len(queue):
        queue["label (correct / wrong / unsure)"], queue["note"] = "", ""
    queue_path = FINAL_DIR / "adjudication_queue.csv"
    if not queue_path.is_file() or pd.read_csv(queue_path, dtype=str).fillna("")["label (correct / wrong / unsure)"].eq("").all():
        queue.to_csv(queue_path, index=False)
    items.merge(pending.groupby("identity")["arm"].agg(lambda a: " | ".join(sorted(set(a)))).rename("arms"), on="identity")[
        ["item_id", "qid", "arms", "identity"]].to_csv(FINAL_DIR / "adjudication_key_SEALED.csv", index=False)

    # Per-arm summary.
    def arm_summary(frame):
        ans, unans = frame[frame["answerable"]], frame[~frame["answerable"]]
        return {"answerable": len(ans), "S-v1 correct": int(ans["s1_correct"].sum()), "accuracy": float(ans["s1_correct"].mean()),
                "unanswerable": len(unans), "correct abstentions": int(unans["abstained"].sum()),
                "citation validity": float(frame["citation_valid"].mean()), "S-v2 correct (auto + owner)": int((ans["v2_final"] == "correct").sum()),
                "S-v2 pending": int((ans["v2_final"] == "pending").sum()), "pages read (mean)": float(frame["pages_read"].mean()),
                "prompt tokens (mean)": float(frame["prompt_tokens"].mean()), "cost USD": float(frame["cost_usd"].sum()),
                "latency p50 s": float(frame["latency_s"].median()), "latency p90 s": float(frame["latency_s"].quantile(0.9))}

    ARMS = ["B1", "T1", "A1", "F8"]
    arm_table = pd.DataFrame({arm: arm_summary(scores[scores["arm"] == arm]) for arm in ARMS}).T
    display(arm_table.round(4))

    # Statistics: document-cluster bootstrap and Monte Carlo sign-flip test.
    def correct(arm, column="s1_correct", subset=None):
        frame = scores[(scores["arm"] == arm) & scores["answerable"]].set_index("qid").sort_index()
        if subset is not None:
            frame = frame[frame.index.isin(subset)]
        return frame[column].astype(float)

    def doc_sums(values):
        docs = pool_q.loc[values.index, "doc_id"].to_numpy()
        order = sorted(set(docs))
        return np.array([values[docs == d].sum() for d in order]), np.array([(docs == d).sum() for d in order])

    def accuracy_ci(values, seed=20261202, draws=10_000):
        sums, counts = doc_sums(values)
        picks = np.random.default_rng(seed).integers(0, len(sums), size=(draws, len(sums)))
        boot = sums[picks].sum(axis=1) / counts[picks].sum(axis=1)
        return {"n": len(values), "correct": int(values.sum()), "accuracy": float(values.mean()), "ci95": [float(x) for x in np.percentile(boot, [2.5, 97.5])],
                "documents": len(sums)}

    def contrast(treatment, control, seed=20261202, flip_seed=20261203):
        diff = (treatment - control).sort_index()
        sums, counts = doc_sums(diff)
        picks = np.random.default_rng(seed).integers(0, len(sums), size=(10_000, len(sums)))
        boot = sums[picks].sum(axis=1) / counts[picks].sum(axis=1)
        signs = np.random.default_rng(flip_seed).choice(np.array([-1.0, 1.0], dtype=np.float32), size=(100_000, len(sums)))
        flipped = signs @ sums.astype(np.float32)
        return {"n": len(diff), "delta": float(diff.mean()), "ci95": [float(x) for x in np.percentile(boot, [2.5, 97.5])],
                "p05_one_sided": float(np.percentile(boot, 5)), "sign_flip_p": float((1 + np.sum(flipped >= sums.sum() - 1e-6)) / (1 + len(flipped))),
                "fixed": int(((treatment == 1) & (control == 0)).sum()), "broken": int(((treatment == 0) & (control == 1)).sum()), "documents": len(sums)}

    born_digital = pool_pages.groupby("doc_id")["native_words"].apply(lambda w: float((w >= 30).mean()) >= 0.8)
    pool_q["born_digital"] = pool_q["doc_id"].map(born_digital)
    pool_q["core_scope"] = pool_q["born_digital"] & pool_q["doc_type"].isin(REPORT_TYPES)
    CORE = set(pool_q.index[pool_q["core_scope"]])

    C1 = {"overall": accuracy_ci(correct("B1")), "core scope": accuracy_ci(correct("B1", subset=CORE)),
          "development estimate (E8 L-B1)": {"correct": 46, "n": 102, "accuracy": 46 / 102}}
    C2 = contrast(correct("B1"), correct("T1"))
    C2["verdict"] = ("visual_retrieval_confirmed" if C2["delta"] > 0 and C2["p05_one_sided"] > 0 and C2["sign_flip_p"] <= 0.05
                     else "reversed" if C2["ci95"][1] < 0 else "visual_retrieval_not_confirmed")
    C3 = contrast(correct("A1"), correct("B1"))
    C3["verdict"] = "a1_gain_supported" if C3["p05_one_sided"] > 0 and C3["sign_flip_p"] <= 0.05 else "a1_gain_not_supported"
    display(pd.DataFrame({"C1 B1 overall": C1["overall"], "C1 B1 core scope": C1["core scope"]}).T)
    display(pd.DataFrame({"C2 B1 - T1": C2, "C3 A1 - B1": C3}).T)

    # S-v2: the exact range of each contrast over every consistent labelling of the pending items (each identity labelled once).
    def v2_range(t_arm, c_arm):
        t = scores[(scores["arm"] == t_arm) & scores["answerable"]].set_index("qid")
        c = scores[(scores["arm"] == c_arm) & scores["answerable"]].set_index("qid").loc[t.index]
        fixed = ((t["v2_final"] == "correct").astype(int) - (c["v2_final"] == "correct").astype(int)).sum()
        weight = Counter(t.loc[t["v2_final"] == "pending", "identity"])
        weight.subtract(Counter(c.loc[c["v2_final"] == "pending", "identity"]))
        low = fixed + sum(min(0, w) for w in weight.values())
        high = fixed + sum(max(0, w) for w in weight.values())
        n = len(t) - int(((t["v2_final"] == "unsure") | (c["v2_final"] == "unsure")).sum())
        return [float(low / n), float(high / n)]

    S_V2 = {"B1 - T1": v2_range("B1", "T1"), "A1 - B1": v2_range("A1", "B1"), "pending items": len(queue)}

    # Secondary: evidence retrieval at k = 4 (annotations used for scoring only), ColQwen2 vs BM25.
    with_evidence = [q for q in pool_q.index if pool_q.at[q, "answerable"] and pool_q.at[q, "evidence_pages"]]
    retrieval = pd.DataFrame([{"qid": q, "V_recall4": len(set(pool_q.at[q, "evidence_pages"]) & set(pool_rankings.at[q, "V_pages"][:4])) / len(set(pool_q.at[q, "evidence_pages"])),
                               "A_recall4": len(set(pool_q.at[q, "evidence_pages"]) & set(pool_rankings.at[q, "A_pages"][:4])) / len(set(pool_q.at[q, "evidence_pages"])),
                               "V_complete4": set(pool_q.at[q, "evidence_pages"]) <= set(pool_rankings.at[q, "V_pages"][:4]),
                               "A_complete4": set(pool_q.at[q, "evidence_pages"]) <= set(pool_rankings.at[q, "A_pages"][:4])} for q in with_evidence]).set_index("qid")
    RETRIEVAL = {"questions": len(retrieval), "V recall@4": float(retrieval["V_recall4"].mean()), "BM25 recall@4": float(retrieval["A_recall4"].mean()),
                 "V complete@4": float(retrieval["V_complete4"].mean()), "BM25 complete@4": float(retrieval["A_complete4"].mean()),
                 "V - BM25 recall@4": contrast(retrieval["V_recall4"], retrieval["A_recall4"])}

    # Prespecified subgroups (descriptive): B1 and T1 accuracy and their difference.
    ans_q = pool_q[pool_q["answerable"]].copy()
    ans_q["pages"] = ans_q["doc_id"].map(pool_documents.set_index("doc_id")["pages"])
    groupings = {
        "document type": ans_q["doc_type"],
        "core scope": ans_q["core_scope"].map({True: "core", False: "other"}),
        "document length": pd.cut(ans_q["pages"], [0, 50, 200, 10_000], labels=["<= 50", "51-200", "> 200"]).astype(str),
        "evidence span": ans_q["evidence_pages"].map(lambda e: "none annotated" if not e else "1" if len(e) == 1 else "2-3" if len(e) <= 3 else "4+"),
        "answer format": ans_q["answer_format"],
        "born-digital": ans_q["born_digital"].map({True: "born-digital", False: "image-heavy"}),
    }
    b1c, t1c = correct("B1"), correct("T1")
    sub_rows = []
    for name, labels in groupings.items():
        for value, index in labels.groupby(labels).groups.items():
            sub_rows.append({"grouping": name, "group": value, "n": len(index), "B1": float(b1c.loc[index].mean()), "T1": float(t1c.loc[index].mean()),
                             "B1 - T1": float((b1c.loc[index] - t1c.loc[index]).mean())})
    sources = ans_q["evidence_sources"].explode().dropna()
    for value, index in sources.groupby(sources).groups.items():
        index = sorted(set(index))
        sub_rows.append({"grouping": "evidence source (overlapping)", "group": value, "n": len(index), "B1": float(b1c.loc[index].mean()),
                         "T1": float(t1c.loc[index].mean()), "B1 - T1": float((b1c.loc[index] - t1c.loc[index]).mean())})
    subgroups = pd.DataFrame(sub_rows)
    display(subgroups.round(3))

    final_summary = {"run": settings["run_id"], "arms": json.loads(arm_table.to_json(orient="index")), "C1": C1, "C2": C2, "C3": C3,
                     "S_v2_ranges": S_V2, "retrieval": RETRIEVAL, "subgroups": json.loads(subgroups.to_json(orient="records")),
                     "triggered": int(pool_rankings["triggered"].sum()), "questions": len(pool_q), "documents": int(pool_q["doc_id"].nunique()),
                     "observed_cost_usd": float(paid["cost_usd"].sum())}
    scores.drop(columns=["prediction", "identity"]).to_csv(FINAL_DIR / "evaluation_per_question.csv", index=False)
    atomic_write_bytes(FINAL_DIR / "evaluation_summary.json", json.dumps(final_summary, indent=1, default=str).encode())
    if FINAL_FROZEN in EVAL_RUN.parents:
        frozen_summary = json.loads((FINAL_FROZEN / "tables" / "evaluation_summary.json").read_text())
        assert json.loads(json.dumps(final_summary, default=str)) == frozen_summary, "the final evaluation no longer reproduces its frozen copy"
        print("Frozen final evaluation reproduced exactly.")
    print(f"C1: B1 accuracy {C1['overall']['accuracy']:.3f} {C1['overall']['ci95']}. C2: {C2['verdict']} (delta {C2['delta']:+.3f}). "
          f"C3: {C3['verdict']} (delta {C3['delta']:+.3f}). S-v2 items for the owner: {len(queue)}.")

No completed final run yet: the evaluation waits for the paid run.


## Results and interpretation

*Written after the run, beside the evidence above.*